In [ ]:
"""
================================================================================
CRESTA MLE INTERVIEW — PRACTICAL CODING PROBLEM
================================================================================
DOMAIN:  Agent Behavior Detection & Outcome Linking
TIME:    60-90 minutes
LEVEL:   Mid-Senior MLE

SCENARIO
--------
You are an MLE on Cresta's Coaching & Behavioral Guidance team. The core
insight behind Cresta's product: certain agent BEHAVIORS during a call
(e.g., "used the customer's name", "offered a retention discount") directly
CAUSE better outcomes (higher CSAT, lower churn, faster resolution).

Your job is to:
  1. Detect which behaviors an agent performed during a call
  2. Link those behaviors to measurable outcomes
  3. Identify which behaviors have the strongest impact
  4. Generate coaching recommendations for individual agents

This powers two Cresta features:
  - Real-time hints: "Try offering a loyalty discount now" (during the call)
  - Post-call coaching: "You resolved 80% of billing calls but only used
    empathy statements in 30% — top agents use them in 75%"

YOUR TASK (4 parts)
-------------------
Part 1: Build behavior detectors from transcripts        [15 min]
Part 2: Link behaviors to outcomes                       [15 min]
Part 3: Measure behavior impact (correlation + uplift)   [20 min]
Part 4: Generate coaching recommendations                [15 min]
================================================================================
"""

import re
import json
from typing import Dict, List, Tuple, Optional, Set
from dataclasses import dataclass, field
from collections import Counter, defaultdict


# ─────────────────────────────────────────────────────────────────────────────
# DATA: 20 conversations with outcomes (do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class CallRecord:
    call_id: str
    agent_id: str
    team: str
    transcript: List[Dict]        # [{"role": "agent"/"customer", "text": "..."}]
    outcome: Dict                 # {"csat": 1-5, "resolved": bool, "repeat_call": bool, "handle_time_sec": int}


CALLS = [
    CallRecord("c001", "agent_sarah", "billing", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, my name is Sarah. How can I help you today?"},
        {"role": "customer", "text": "Hi Sarah, I was charged $47.99 for something I didn't order."},
        {"role": "agent", "text": "I'm sorry to hear about that unexpected charge, that must be really frustrating. Let me pull up your account right away."},
        {"role": "agent", "text": "I can see the issue — this was added during a system migration. I'm going to remove it and issue a full refund of $47.99 to your next billing cycle."},
        {"role": "agent", "text": "So to confirm, I've removed the charge and you'll see a $47.99 credit on your next statement. Is there anything else I can help with?"},
        {"role": "customer", "text": "No, that's great. Thanks Sarah!"},
        {"role": "agent", "text": "You're welcome! Thank you for being a loyal customer. Have a great day."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 180}),

    CallRecord("c002", "agent_tony", "billing", [
        {"role": "agent", "text": "Meridian support."},
        {"role": "customer", "text": "My bill is wrong again. Third time calling about this."},
        {"role": "agent", "text": "Account number?"},
        {"role": "customer", "text": "AC-6681200. Can someone actually fix this?"},
        {"role": "agent", "text": "Let me look... yeah there's an add-on that keeps coming back. I'll remove it again."},
        {"role": "customer", "text": "You'll remove it AGAIN? That's what the last two agents said."},
        {"role": "agent", "text": "I understand. I'll escalate to tier 2. Someone will call you."},
        {"role": "agent", "text": "Anything else?"},
        {"role": "customer", "text": "Just fix it."},
        {"role": "agent", "text": "Will do. Bye."},
    ], {"csat": 1, "resolved": False, "repeat_call": True, "handle_time_sec": 120}),

    CallRecord("c003", "agent_lisa", "retention", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless! I'm Lisa, how are you doing today?"},
        {"role": "customer", "text": "I want to cancel. Switching to Apex Mobile."},
        {"role": "agent", "text": "I'm sorry to hear you're thinking about leaving. May I ask what's driving the switch?"},
        {"role": "customer", "text": "Price. I'm paying $89 and Apex is $59."},
        {"role": "agent", "text": "I completely understand — nobody wants to feel like they're overpaying. You've been with us for three years and I really value that loyalty."},
        {"role": "agent", "text": "I can offer you our loyalty rate of $62 per month, locked in for 12 months. That's close to the Apex price and you keep your current number and device plan."},
        {"role": "customer", "text": "That's actually not bad. I'll take it."},
        {"role": "agent", "text": "Wonderful! I've applied the new rate starting today. You'll see it on your next bill. Is there anything else I can help with?"},
        {"role": "customer", "text": "No, thanks Lisa."},
        {"role": "agent", "text": "My pleasure! Have a wonderful evening."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 210}),

    CallRecord("c004", "agent_sarah", "billing", [
        {"role": "agent", "text": "Thank you for calling Meridian, my name is Sarah. How can I help?"},
        {"role": "customer", "text": "I need to understand my bill. It seems too high."},
        {"role": "agent", "text": "I'd be happy to walk through your bill with you. Can I get your account number?"},
        {"role": "customer", "text": "AC-4412890."},
        {"role": "agent", "text": "Thanks. Looking at your bill, I can see you're on the Plus plan at $69 per month. There's also a $15 international calling add-on and a $9.99 late fee from last month."},
        {"role": "agent", "text": "The total comes to $93.99. Would you like me to go through each charge in detail?"},
        {"role": "customer", "text": "No, that makes sense now. Can you remove the international calling? I don't need it anymore."},
        {"role": "agent", "text": "Absolutely, I've removed the international calling add-on. Your next bill will be $69 plus tax. And I want to mention — if you set up AutoPay, you can save an extra $5 per month."},
        {"role": "customer", "text": "Oh nice, I'll do that. Thanks for explaining everything."},
        {"role": "agent", "text": "You're welcome! I'm glad I could clarify things. Have a great day."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 195}),

    CallRecord("c005", "agent_tony", "technical", [
        {"role": "agent", "text": "Meridian support, Tony."},
        {"role": "customer", "text": "My internet is really slow."},
        {"role": "agent", "text": "Have you tried restarting your modem?"},
        {"role": "customer", "text": "Yes, twice."},
        {"role": "agent", "text": "Okay let me run a diagnostic... speeds look fine on our end. It's probably a WiFi issue. Move your router somewhere central."},
        {"role": "customer", "text": "I've already done that. It's been slow for a week."},
        {"role": "agent", "text": "I'll put in a ticket for a technician. Should be 24-48 hours."},
        {"role": "customer", "text": "A week of slow internet and now I have to wait two more days?"},
        {"role": "agent", "text": "That's the standard timeframe. Anything else?"},
        {"role": "customer", "text": "No."},
        {"role": "agent", "text": "Okay, bye."},
    ], {"csat": 2, "resolved": False, "repeat_call": True, "handle_time_sec": 150}),

    CallRecord("c006", "agent_jessica", "general", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, I'm Jessica. How can I assist you today?"},
        {"role": "customer", "text": "I need to set up parental controls for my kids."},
        {"role": "agent", "text": "Great question! We have a free app called Meridian Safe that gives you full control. Have you heard of it before?"},
        {"role": "customer", "text": "No, how does it work?"},
        {"role": "agent", "text": "It's really easy — you download Meridian Safe from your app store and log in with your Meridian credentials. From there you can block website categories like gaming or social media, set per-device time limits, and even schedule automatic bedtime shutoffs."},
        {"role": "agent", "text": "The best part is it works on every device connected to your home WiFi — phones, tablets, gaming consoles, everything."},
        {"role": "customer", "text": "That sounds perfect. Does it work on cellular data too?"},
        {"role": "agent", "text": "That's a great question. The controls only apply on your home WiFi network, so if the kids switch to cellular data, the blocks won't be active there. But for home use, it covers everything."},
        {"role": "customer", "text": "Okay, I'll set it up tonight. Thanks!"},
        {"role": "agent", "text": "You're welcome! And if you run into any issues during setup, don't hesitate to call us back. We're always here to help. Have a wonderful evening!"},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 165}),

    CallRecord("c007", "agent_tony", "billing", [
        {"role": "agent", "text": "Meridian."},
        {"role": "customer", "text": "When is my payment due?"},
        {"role": "agent", "text": "The 15th. Anything else?"},
        {"role": "customer", "text": "What if I'm late?"},
        {"role": "agent", "text": "$9.99 fee after 5 days."},
        {"role": "customer", "text": "Okay. Thanks."},
        {"role": "agent", "text": "Yep."},
    ], {"csat": 3, "resolved": True, "repeat_call": False, "handle_time_sec": 45}),

    CallRecord("c008", "agent_lisa", "retention", [
        {"role": "agent", "text": "Thank you for calling Meridian, I'm Lisa. What can I do for you today?"},
        {"role": "customer", "text": "I want to cancel my service."},
        {"role": "agent", "text": "I'm sorry to hear that. I'd love to understand what's going on — can you share what's prompting the decision?"},
        {"role": "customer", "text": "I just don't use it enough to justify the cost."},
        {"role": "agent", "text": "I hear you — paying for something you don't fully use is frustrating. Let me look at your usage and see if there's a plan that better fits what you actually need."},
        {"role": "agent", "text": "Looking at your account, you're on the Premium plan at $99 but your usage fits the Basic plan at $49. That would save you $50 a month. Would you like me to switch you?"},
        {"role": "customer", "text": "Oh wow, yeah that would be much better."},
        {"role": "agent", "text": "Done! I've downgraded you to the Basic plan at $49 per month, effective today. You'll see the change on your next bill. Is there anything else?"},
        {"role": "customer", "text": "No, that's perfect. I was ready to cancel but this works."},
        {"role": "agent", "text": "I'm so glad we found a solution that works for you! Thank you for staying with us. Have a great day!"},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 200}),

    CallRecord("c009", "agent_david", "billing", [
        {"role": "agent", "text": "Hello, Meridian Wireless, David speaking. How can I help you?"},
        {"role": "customer", "text": "I have two issues — I need to update my address and I have a weird charge."},
        {"role": "agent", "text": "Sure, I'll help with both. Let's start with the address change. What's your new address?"},
        {"role": "customer", "text": "445 Birch Lane, Portland."},
        {"role": "agent", "text": "Updated. Now about the charge — what are you seeing?"},
        {"role": "customer", "text": "$14.99 for Device Protection Plus. I never added that."},
        {"role": "agent", "text": "I see it was added through the app. I've removed it and refunded this month. For the prior months, I need to submit a dispute — you'll hear back in 5-7 days. Reference number is RF-44217."},
        {"role": "agent", "text": "To summarize: address updated, protection plan removed and refunded, dispute submitted for the remaining balance. Anything else?"},
        {"role": "customer", "text": "That covers it. Thanks David."},
        {"role": "agent", "text": "You're welcome. Good luck with the move!"},
    ], {"csat": 4, "resolved": True, "repeat_call": False, "handle_time_sec": 175}),

    CallRecord("c010", "agent_tony", "technical", [
        {"role": "agent", "text": "Meridian support."},
        {"role": "customer", "text": "My internet is completely down. No connection at all."},
        {"role": "agent", "text": "Is your modem plugged in?"},
        {"role": "customer", "text": "Of course it is. I'm not an idiot."},
        {"role": "agent", "text": "Just checking. Let me see if there's an outage... yeah there's one in your area. Should be fixed in a few hours."},
        {"role": "customer", "text": "A few hours? I'm working from home!"},
        {"role": "agent", "text": "Not much we can do during an outage. Check status.meridianwireless.com for updates."},
        {"role": "customer", "text": "This is ridiculous."},
        {"role": "agent", "text": "Sorry. Anything else?"},
        {"role": "customer", "text": "No."},
    ], {"csat": 1, "resolved": False, "repeat_call": True, "handle_time_sec": 95}),

    CallRecord("c011", "agent_sarah", "billing", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, my name is Sarah. How can I help you today?"},
        {"role": "customer", "text": "I got a late fee but I paid on time. I have the bank confirmation."},
        {"role": "agent", "text": "That sounds frustrating — let me look into this for you right away."},
        {"role": "agent", "text": "I can see your payment was received on the 16th, one day after the due date. However, I can see from the bank timestamp that you initiated it on the 14th. The delay was on our processing side."},
        {"role": "agent", "text": "Since this wasn't your fault, I'm waiving the $9.99 late fee right now. You'll see the credit on your next statement."},
        {"role": "customer", "text": "Thank you, I appreciate you looking into it."},
        {"role": "agent", "text": "Of course! And just a tip — if you set up AutoPay, payments process automatically on the due date so you never have to worry about timing. Would you like me to set that up?"},
        {"role": "customer", "text": "Sure, let's do it."},
        {"role": "agent", "text": "All set! AutoPay is enabled and you'll also get a $5 monthly discount. To confirm — late fee waived, AutoPay enabled with $5 discount. Anything else?"},
        {"role": "customer", "text": "No, that's perfect. Thanks Sarah!"},
        {"role": "agent", "text": "You're welcome! Have a wonderful day."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 210}),

    CallRecord("c012", "agent_jessica", "general", [
        {"role": "agent", "text": "Thank you for calling Meridian, I'm Jessica. How can I help?"},
        {"role": "customer", "text": "I want to know about your referral program."},
        {"role": "agent", "text": "Great question! Our referral program is really simple. You share your unique referral code — you can find it in the app under account settings — and when someone signs up with it, you both get a $50 account credit after their first 30 days."},
        {"role": "customer", "text": "Is there a limit on how many people I can refer?"},
        {"role": "agent", "text": "No limit at all! And I should mention — top referrers qualify for our Champions Program which includes perks like a dedicated support line and early access to new products."},
        {"role": "customer", "text": "That's awesome, I'll start sharing it."},
        {"role": "agent", "text": "That's great to hear! Your referral code is already in your app. Is there anything else I can help with today?"},
        {"role": "customer", "text": "Nope, that's all. Thank you!"},
        {"role": "agent", "text": "You're welcome! Happy referring, and have a great day!"},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 130}),

    CallRecord("c013", "agent_david", "technical", [
        {"role": "agent", "text": "Meridian Wireless, David here. What's going on?"},
        {"role": "customer", "text": "WiFi keeps dropping on my laptop but works fine on my phone."},
        {"role": "agent", "text": "Interesting — that tells me it's likely a device-specific issue rather than a network problem. What kind of laptop do you have?"},
        {"role": "customer", "text": "A MacBook Pro."},
        {"role": "agent", "text": "Got it. A few things to try: First, forget the WiFi network on your MacBook and reconnect. Second, make sure your MacBook is using the 5 GHz band — it's faster and less congested. You can check in the WiFi settings."},
        {"role": "customer", "text": "How do I switch to 5 GHz?"},
        {"role": "agent", "text": "Go to System Settings, then WiFi. You should see two versions of your network — one regular and one with 5G or 5GHz in the name. Connect to that one. If you only see one, your router might be broadcasting both on the same name — I can walk you through splitting them if you'd like."},
        {"role": "customer", "text": "Let me try forgetting and reconnecting first... okay, it reconnected. Let me test it."},
        {"role": "agent", "text": "Take your time."},
        {"role": "customer", "text": "Seems faster already. I think that did it."},
        {"role": "agent", "text": "Excellent! If it drops again, try the 5 GHz band switch I mentioned. And if the problem comes back, call us and we can dig deeper. Is there anything else?"},
        {"role": "customer", "text": "No, that's great. Thanks David."},
        {"role": "agent", "text": "Happy to help! Have a good one."},
    ], {"csat": 4, "resolved": True, "repeat_call": False, "handle_time_sec": 240}),

    CallRecord("c014", "agent_sarah", "retention", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, I'm Sarah. How can I help you today?"},
        {"role": "customer", "text": "I want to cancel. The service is too expensive."},
        {"role": "agent", "text": "I'm sorry to hear that. I know how important it is to feel like you're getting good value. Can I take a look at your account to see if there's a plan that fits your budget better?"},
        {"role": "customer", "text": "I already have the cheapest plan."},
        {"role": "agent", "text": "I understand. Let me check if there are any promotions or loyalty offers I can apply to your account... I can see you've been with us for 18 months. I can offer a $10 per month discount for the next 6 months. That would bring your Basic plan from $49 to $39."},
        {"role": "customer", "text": "Hmm, $39 is more manageable. But will it go back up?"},
        {"role": "agent", "text": "After 6 months it returns to $49. But at that point, I'd encourage you to call us again — there may be new promotions or loyalty tiers available. We want to keep you as a customer."},
        {"role": "customer", "text": "Okay, I'll take the discount."},
        {"role": "agent", "text": "Great! I've applied the $10 discount starting today. To confirm — your bill will be $39 per month for the next 6 months. I'll send a confirmation email. Is there anything else?"},
        {"role": "customer", "text": "No, thanks Sarah."},
        {"role": "agent", "text": "Thank you for staying with Meridian! Have a great day."},
    ], {"csat": 4, "resolved": True, "repeat_call": False, "handle_time_sec": 220}),

    CallRecord("c015", "agent_tony", "general", [
        {"role": "agent", "text": "Meridian."},
        {"role": "customer", "text": "How do I check my data usage?"},
        {"role": "agent", "text": "The app. Under usage."},
        {"role": "customer", "text": "Which app?"},
        {"role": "agent", "text": "The Meridian app. Download it and log in."},
        {"role": "customer", "text": "Okay... anything else I should know?"},
        {"role": "agent", "text": "Nah, it's pretty self-explanatory. Bye."},
    ], {"csat": 2, "resolved": True, "repeat_call": False, "handle_time_sec": 40}),

    CallRecord("c016", "agent_jessica", "technical", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, I'm Jessica. How can I help?"},
        {"role": "customer", "text": "My TV box is frozen and won't respond to the remote."},
        {"role": "agent", "text": "Oh no, that's annoying! Let's get that sorted out. First, let's try a simple restart — unplug the TV box from power, wait about 30 seconds, and plug it back in."},
        {"role": "customer", "text": "Okay, doing it now... it's restarting."},
        {"role": "agent", "text": "Perfect, give it about a minute to fully boot up."},
        {"role": "customer", "text": "It's back! And the remote is working now."},
        {"role": "agent", "text": "Wonderful! Sometimes these boxes just need a fresh start. If it happens frequently, it might need a firmware update — you can check that in Settings > System > Software Update. Or we can send a replacement box if it keeps freezing."},
        {"role": "customer", "text": "I'll check the firmware. Thanks for the quick fix!"},
        {"role": "agent", "text": "My pleasure! Don't hesitate to call back if it happens again. Have a great day!"},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 140}),

    CallRecord("c017", "agent_david", "billing", [
        {"role": "agent", "text": "Meridian Wireless, David speaking. What can I help you with?"},
        {"role": "customer", "text": "I want to switch from monthly billing to annual if there's a discount."},
        {"role": "agent", "text": "That's a smart question. Unfortunately, we don't currently offer annual billing with a discount. All our plans are month-to-month."},
        {"role": "customer", "text": "That's disappointing. Is there any other way to save?"},
        {"role": "agent", "text": "There are a couple options. AutoPay gives you a $5 per month discount. And our referral program gives you a $50 credit for each friend who signs up."},
        {"role": "customer", "text": "I already have AutoPay. I'll try the referral thing."},
        {"role": "agent", "text": "Great — your referral code is in the app under account settings. Each successful referral gets you and your friend $50 credit. Is there anything else?"},
        {"role": "customer", "text": "No, thanks."},
        {"role": "agent", "text": "Have a good day!"},
    ], {"csat": 3, "resolved": True, "repeat_call": False, "handle_time_sec": 110}),

    CallRecord("c018", "agent_lisa", "billing", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, I'm Lisa. How can I help you today?"},
        {"role": "customer", "text": "I'm confused about my bill. There are charges I don't understand."},
        {"role": "agent", "text": "I'm sorry for the confusion — bills can definitely be overwhelming with all the line items. Let me walk through it with you step by step."},
        {"role": "agent", "text": "Your base plan is Plus at $69. Then there's a $15 international calling add-on, a $5 equipment rental fee, and taxes of $7.43. Total is $96.43."},
        {"role": "customer", "text": "I don't think I need international calling anymore."},
        {"role": "agent", "text": "No problem, I can remove that right now. That brings your next bill down to $81.43. Much better, right?"},
        {"role": "customer", "text": "Yes! And I also want to mention — the last agent I spoke to was really rude."},
        {"role": "agent", "text": "I'm really sorry to hear about that experience. That's not the standard we hold ourselves to. I'll make a note on your account so a manager can follow up. Your feedback helps us improve."},
        {"role": "customer", "text": "Thank you, I appreciate that."},
        {"role": "agent", "text": "Of course! To summarize — I've removed international calling, saving you $15 per month, and I've logged your feedback about the previous interaction. Is there anything else?"},
        {"role": "customer", "text": "No, that's everything. Thanks Lisa!"},
        {"role": "agent", "text": "You're welcome! Have a lovely day."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 195}),

    CallRecord("c019", "agent_tony", "retention", [
        {"role": "agent", "text": "Meridian, Tony speaking."},
        {"role": "customer", "text": "I want to cancel."},
        {"role": "agent", "text": "Why?"},
        {"role": "customer", "text": "Too expensive. $99 a month is too much."},
        {"role": "agent", "text": "I can switch you to the Basic plan for $49. Want me to do that?"},
        {"role": "customer", "text": "What's the difference?"},
        {"role": "agent", "text": "Basic is 100 Mbps instead of 1 Gbps. Probably fine unless you're doing a lot of streaming."},
        {"role": "customer", "text": "Fine, switch me."},
        {"role": "agent", "text": "Done. Anything else?"},
        {"role": "customer", "text": "No."},
        {"role": "agent", "text": "Bye."},
    ], {"csat": 2, "resolved": True, "repeat_call": False, "handle_time_sec": 75}),

    CallRecord("c020", "agent_sarah", "technical", [
        {"role": "agent", "text": "Thank you for calling Meridian Wireless, my name is Sarah. How can I help you?"},
        {"role": "customer", "text": "My internet is slow and I'm paying for 300 Mbps."},
        {"role": "agent", "text": "I understand how frustrating that is, especially when you're paying for fast speeds. Let me run a diagnostic on your line."},
        {"role": "agent", "text": "Okay, I can see you're getting about 180 Mbps on a wired connection. That's below where it should be. Let me check the line quality..."},
        {"role": "agent", "text": "There's some signal degradation that could be causing the drop. I'm going to do two things: first, I'll push a firmware update to your modem remotely. Second, I'll schedule a technician visit to inspect the line. Which works better — morning or afternoon?"},
        {"role": "customer", "text": "Afternoon, please."},
        {"role": "agent", "text": "Done — I've scheduled a technician for Thursday afternoon between 1 and 5 PM. You'll get a confirmation text. In the meantime, the firmware update should help — you might see improvement within the hour."},
        {"role": "customer", "text": "Great, thank you for being so thorough."},
        {"role": "agent", "text": "Absolutely! I also want to mention — since you're getting below your plan speed, I'm applying a $15 credit to your account for this month. You shouldn't pay full price for speeds you're not getting."},
        {"role": "customer", "text": "Wow, I didn't even ask for that. Thank you!"},
        {"role": "agent", "text": "Of course! To confirm — firmware updating, technician Thursday afternoon, and $15 credit applied. If speeds don't improve after the firmware update, the technician will sort it out. Anything else?"},
        {"role": "customer", "text": "No, you've been amazing. Thanks Sarah!"},
        {"role": "agent", "text": "Thank you! Have a wonderful day."},
    ], {"csat": 5, "resolved": True, "repeat_call": False, "handle_time_sec": 270}),
]


# ─────────────────────────────────────────────────────────────────────────────
# BEHAVIOR DEFINITIONS — the behaviors to detect
# ─────────────────────────────────────────────────────────────────────────────

BEHAVIORS = {
    "professional_greeting": "Agent introduced themselves by name and offered to help",
    "used_customer_name": "Agent used the customer's first name during the call",
    "empathy_statement": "Agent acknowledged the customer's feelings or frustration",
    "proactive_offer": "Agent offered something the customer didn't ask for (discount, tip, additional service)",
    "clear_explanation": "Agent explained something step-by-step or walked the customer through details",
    "action_confirmation": "Agent summarized actions taken before closing the call",
    "warm_closing": "Agent closed with warmth beyond just 'bye' (thanked customer, wished them well, offered future help)",
    "asked_discovery_question": "Agent asked a question to understand the customer's needs or situation",
    "ownership_language": "Agent used first-person commitment language (I'll, I'm going to, Let me)",
    "alternative_offered": "Agent offered an alternative when the first option wasn't ideal",
}


# ─────────────────────────────────────────────────────────────────────────────
# PART 1: BEHAVIOR DETECTION                                       [15 min]
# ─────────────────────────────────────────────────────────────────────────────
#
# Detect which behaviors appear in each conversation. Use simple keyword
# and pattern matching — this is a prototype, not a production model.
# ─────────────────────────────────────────────────────────────────────────────

def detect_behaviors(call: CallRecord) -> Dict[str, bool]:
    """
    For each behavior in BEHAVIORS, detect whether it was performed.

    Returns: {"professional_greeting": True/False, "used_customer_name": True/False, ...}

    Guidelines for detection (keyword/pattern heuristics):
      - professional_greeting: agent says their name ("my name is X" / "I'm X") + offers help
      - used_customer_name: agent utterance contains a customer name (from customer's self-intro)
      - empathy_statement: agent uses phrases like "I understand", "sorry", "frustrating", "I hear you"
      - proactive_offer: agent offers something unprompted (discount, tip, AutoPay, referral)
      - clear_explanation: agent gives multi-step or detailed explanation
      - action_confirmation: agent summarizes ("to confirm", "to summarize", "to recap")
      - warm_closing: agent says "have a great/wonderful day", "thank you for", offers future help
      - asked_discovery_question: agent asks an open question ("can you tell me", "what's", "may I ask")
      - ownership_language: agent uses "I'll", "I'm going to", "Let me", "I've", "I can"
      - alternative_offered: agent proposes a different option or plan
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1: Implement detect_behaviors")


def detect_all(calls: List[CallRecord]) -> Dict[str, Dict[str, bool]]:
    """Run detection on all calls. Returns {call_id: {behavior: bool}}."""
    # TODO: Implement this
    raise NotImplementedError("Part 1b: Implement detect_all")


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: LINK BEHAVIORS TO OUTCOMES                               [15 min]
# ─────────────────────────────────────────────────────────────────────────────

def behavior_outcome_table(
    calls: List[CallRecord],
    detections: Dict[str, Dict[str, bool]],
) -> Dict[str, Dict]:
    """
    For each behavior, compute outcome stats when present vs absent.

    Returns:
    {
        "professional_greeting": {
            "present_count": int,       # calls where behavior was detected
            "absent_count": int,
            "present_avg_csat": float,  # avg CSAT when behavior present
            "absent_avg_csat": float,
            "present_resolve_rate": float,  # % resolved when present
            "absent_resolve_rate": float,
            "present_repeat_rate": float,   # % repeat calls when present
            "absent_repeat_rate": float,
        },
        ...
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2: Implement behavior_outcome_table")


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: MEASURE BEHAVIOR IMPACT                                 [20 min]
# ─────────────────────────────────────────────────────────────────────────────

def compute_behavior_impact(
    calls: List[CallRecord],
    detections: Dict[str, Dict[str, bool]],
) -> List[Dict]:
    """
    Rank behaviors by their impact on CSAT.

    For each behavior, compute:
      - csat_uplift: present_avg_csat - absent_avg_csat
      - resolve_uplift: present_resolve_rate - absent_resolve_rate
      - repeat_reduction: absent_repeat_rate - present_repeat_rate (positive = fewer repeats)

    Returns a list sorted by csat_uplift descending:
    [
        {
            "behavior": str,
            "csat_uplift": float,
            "resolve_uplift": float,
            "repeat_reduction": float,
            "present_count": int,
            "absent_count": int,
            "statistically_meaningful": bool,  # True if both groups have >= 3 calls
        }
    ]

    NOTE: With only 20 calls, correlations are noisy. The
    "statistically_meaningful" flag is a simple check — in production,
    you'd use a proper statistical test (t-test or Mann-Whitney U).
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3a: Implement compute_behavior_impact")


def find_behavior_combinations(
    calls: List[CallRecord],
    detections: Dict[str, Dict[str, bool]],
    min_combo_size: int = 2,
    max_combo_size: int = 3,
) -> List[Dict]:
    """
    BONUS (if time permits): Find combinations of behaviors with the highest
    CSAT uplift. Sometimes it's not one behavior but a combination that matters.

    E.g., "empathy_statement + action_confirmation" together might yield
    higher CSAT than either alone.

    Returns top 5 combinations sorted by avg CSAT descending:
    [
        {
            "behaviors": ["empathy_statement", "action_confirmation"],
            "call_count": int,
            "avg_csat": float,
        }
    ]
    """
    # TODO: Implement this (bonus)
    raise NotImplementedError("Part 3b: Implement find_behavior_combinations")


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: COACHING RECOMMENDATIONS                                [15 min]
# ─────────────────────────────────────────────────────────────────────────────

def agent_behavior_profile(
    calls: List[CallRecord],
    detections: Dict[str, Dict[str, bool]],
) -> Dict[str, Dict]:
    """
    Build a behavior profile for each agent.

    Returns:
    {
        "agent_sarah": {
            "total_calls": int,
            "avg_csat": float,
            "behavior_rates": {"professional_greeting": 0.80, ...},  # % of their calls
            "resolve_rate": float,
        },
        ...
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4a: Implement agent_behavior_profile")


def generate_coaching(
    profiles: Dict[str, Dict],
    impact_ranking: List[Dict],
    team_avg_rates: Optional[Dict[str, float]] = None,
) -> Dict[str, List[str]]:
    """
    Generate 2-3 specific coaching recommendations per agent.

    Logic: compare the agent's behavior rates to the team average. For
    high-impact behaviors where the agent is below average, recommend
    they adopt that behavior.

    If team_avg_rates is None, compute it from all profiles.

    Returns:
    {
        "agent_sarah": [
            "Your empathy statement rate (75%) is above the team average (60%) — keep it up!",
            "Consider adding action confirmations — you use them 50% vs 80% for top agents.",
        ],
        "agent_tony": [
            "Adding a professional greeting with your name could boost CSAT by ~1.2 points.",
            ...
        ],
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4b: Implement generate_coaching")


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("CRESTA MLE — Behavior Detection & Outcome Linking")
    print("=" * 60)

    # ── Part 1 ──────────────────────────────────────────────────────
    print("\n🔍 PART 1: Behavior Detection")
    print("-" * 40)
    detections = {}
    try:
        detections = detect_all(CALLS)
        for call in CALLS[:4]:
            behaviors = detections[call.call_id]
            present = [b for b, v in behaviors.items() if v]
            print(f"  {call.call_id} ({call.agent_id}): {len(present)}/{len(BEHAVIORS)} behaviors")
            print(f"    ✓ {', '.join(present[:5])}")
        print("  ✅ Part 1 passed")
    except NotImplementedError:
        print("  ⬜ Part 1: Not implemented yet")

    # ── Part 2 ──────────────────────────────────────────────────────
    print(f"\n📊 PART 2: Behavior-Outcome Table")
    print("-" * 40)
    try:
        if not detections:
            raise NotImplementedError("Need Part 1")
        table = behavior_outcome_table(CALLS, detections)
        print(f"  {'Behavior':<25s} {'Present':>7s} {'CSAT+':>7s} {'CSAT-':>7s} {'Δ':>7s}")
        for beh in sorted(table.keys()):
            t = table[beh]
            delta = t["present_avg_csat"] - t["absent_avg_csat"]
            print(f"  {beh:<25s} {t['present_count']:>5d}x  {t['present_avg_csat']:>5.1f}  {t['absent_avg_csat']:>5.1f}  {delta:>+5.1f}")
        print("  ✅ Part 2 passed")
    except NotImplementedError:
        print("  ⬜ Part 2: Not implemented yet")

    # ── Part 3 ──────────────────────────────────────────────────────
    print(f"\n📈 PART 3: Behavior Impact Ranking")
    print("-" * 40)
    impact = []
    try:
        if not detections:
            raise NotImplementedError("Need Part 1")
        impact = compute_behavior_impact(CALLS, detections)
        print(f"  Ranked by CSAT uplift:")
        for i, row in enumerate(impact[:7]):
            flag = "✓" if row["statistically_meaningful"] else "~"
            print(f"  {i+1}. [{flag}] {row['behavior']:<25s} CSAT: {row['csat_uplift']:+.2f}  "
                  f"resolve: {row['resolve_uplift']:+.0%}  repeat: {row['repeat_reduction']:+.0%}")
        print("  ✅ Part 3a passed")
    except NotImplementedError:
        print("  ⬜ Part 3a: Not implemented yet")

    try:
        if not detections:
            raise NotImplementedError("Need Part 1")
        combos = find_behavior_combinations(CALLS, detections)
        print(f"\n  Top behavior combos:")
        for combo in combos[:3]:
            print(f"    {' + '.join(combo['behaviors'])}")
            print(f"      {combo['call_count']} calls, avg CSAT={combo['avg_csat']:.1f}")
        print("  ✅ Part 3b passed")
    except NotImplementedError:
        print("  ⬜ Part 3b: Not implemented (bonus)")

    # ── Part 4 ──────────────────────────────────────────────────────
    print(f"\n🎯 PART 4: Coaching Recommendations")
    print("-" * 40)
    try:
        if not detections:
            raise NotImplementedError("Need Part 1")
        profiles = agent_behavior_profile(CALLS, detections)
        coaching = generate_coaching(profiles, impact)
        for agent_id, recs in sorted(coaching.items()):
            csat = profiles[agent_id]["avg_csat"]
            print(f"\n  {agent_id} (avg CSAT: {csat:.1f}):")
            for rec in recs:
                print(f"    → {rec}")
        print("\n  ✅ Part 4 passed")
    except NotImplementedError:
        print("  ⬜ Part 4: Not implemented yet")

    print("\n" + "=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
================================================================================
SIMPLE SOLUTION — Cresta Behavior Detection & Outcome Linking (1 hour)
================================================================================
"""

import re
from typing import Dict, List, Optional
from collections import defaultdict
from itertools import combinations

from cresta_behavior_problem import (
    CallRecord, CALLS, BEHAVIORS,
)


# ═══════════════════════════════════════════════════════════════════════
# PART 1 — behavior detection with keyword heuristics
# ═══════════════════════════════════════════════════════════════════════

def get_agent_texts(call: CallRecord) -> str:
    """Join all agent utterances into one lowercase string."""
    return " ".join(t["text"] for t in call.transcript if t["role"] == "agent").lower()


def get_customer_name(call: CallRecord) -> Optional[str]:
    """Try to extract customer's name from first agent greeting."""
    # Customers often introduce themselves: "Hi Sarah" → the agent's name
    # We want the customer's name used BY the agent later
    # Check if customer says their name or if agent uses a name
    for turn in call.transcript:
        if turn["role"] == "customer":
            m = re.search(r"\bhi (\w+)|i'm (\w+)|my name is (\w+)", turn["text"].lower())
            if m:
                return (m.group(1) or m.group(2) or m.group(3)).capitalize()
    return None


def detect_behaviors(call: CallRecord) -> Dict[str, bool]:
    agent_text = get_agent_texts(call)
    all_text = " ".join(t["text"] for t in call.transcript).lower()
    agent_turns = [t["text"].lower() for t in call.transcript if t["role"] == "agent"]

    results = {}

    # 1. professional_greeting: agent says name + offers help
    first_agent = agent_turns[0] if agent_turns else ""
    has_name = bool(re.search(r"(my name is|i'm) \w+", first_agent))
    has_help = bool(re.search(r"(how can i|what can i|help you)", first_agent))
    results["professional_greeting"] = has_name and has_help

    # 2. used_customer_name: agent uses a person's first name (not their own)
    customer_name = get_customer_name(call)
    if customer_name:
        results["used_customer_name"] = customer_name.lower() in agent_text
    else:
        # Check if agent uses any capitalized name in their text (after greeting)
        results["used_customer_name"] = False

    # 3. empathy_statement
    empathy_phrases = ["sorry", "understand", "frustrat", "i hear you", "i know how",
                       "that must be", "i appreciate", "apologize", "tough", "annoying"]
    results["empathy_statement"] = any(p in agent_text for p in empathy_phrases)

    # 4. proactive_offer: agent offers something unprompted
    offer_phrases = ["i can offer", "i'd like to offer", "i want to mention",
                     "i should mention", "also", "tip", "you might also",
                     "i'm applying a", "i'm also", "discount", "credit to your account",
                     "save you", "save an extra"]
    results["proactive_offer"] = any(p in agent_text for p in offer_phrases)

    # 5. clear_explanation: agent gives multi-step or detailed explanation
    explanation_signals = ["first", "second", "step", "here's how", "let me walk",
                           "let me explain", "go to", "from there", "what happens is"]
    results["clear_explanation"] = any(p in agent_text for p in explanation_signals)

    # 6. action_confirmation: agent summarizes before closing
    confirm_phrases = ["to confirm", "to summarize", "to recap", "so to confirm",
                       "let me confirm", "just to confirm"]
    results["action_confirmation"] = any(p in agent_text for p in confirm_phrases)

    # 7. warm_closing: warm ending
    last_two = " ".join(agent_turns[-2:]) if len(agent_turns) >= 2 else (agent_turns[-1] if agent_turns else "")
    warm_phrases = ["have a great", "have a wonderful", "have a good", "have a lovely",
                    "thank you for", "don't hesitate", "always here to help",
                    "happy referring", "we're always"]
    results["warm_closing"] = any(p in last_two for p in warm_phrases)

    # 8. asked_discovery_question
    question_phrases = ["can you tell me", "may i ask", "what's driving",
                        "what's going on", "what are you seeing", "can you share",
                        "what kind of", "which works better"]
    results["asked_discovery_question"] = any(p in agent_text for p in question_phrases)

    # 9. ownership_language: first-person commitment
    ownership_phrases = ["i'll", "i'm going to", "let me", "i've", "i can", "i will"]
    results["ownership_language"] = any(p in agent_text for p in ownership_phrases)

    # 10. alternative_offered
    alt_phrases = ["instead", "another option", "alternatively", "we could also",
                   "how about", "would you like me to switch", "loyalty rate",
                   "downgrade", "different plan", "better fit"]
    results["alternative_offered"] = any(p in agent_text for p in alt_phrases)

    return results


def detect_all(calls: List[CallRecord]) -> Dict[str, Dict[str, bool]]:
    return {call.call_id: detect_behaviors(call) for call in calls}


# ═══════════════════════════════════════════════════════════════════════
# PART 2 — link behaviors to outcomes
# ═══════════════════════════════════════════════════════════════════════

def behavior_outcome_table(calls, detections):
    table = {}

    for behavior in BEHAVIORS:
        present_csats, absent_csats = [], []
        present_resolved, absent_resolved = [], []
        present_repeat, absent_repeat = [], []

        for call in calls:
            det = detections[call.call_id]
            csat = call.outcome["csat"]
            resolved = call.outcome["resolved"]
            repeat = call.outcome["repeat_call"]

            if det[behavior]:
                present_csats.append(csat)
                present_resolved.append(resolved)
                present_repeat.append(repeat)
            else:
                absent_csats.append(csat)
                absent_resolved.append(resolved)
                absent_repeat.append(repeat)

        def safe_avg(lst, default=0):
            return sum(lst) / len(lst) if lst else default

        def safe_rate(bools, default=0):
            return sum(bools) / len(bools) if bools else default

        table[behavior] = {
            "present_count": len(present_csats),
            "absent_count": len(absent_csats),
            "present_avg_csat": round(safe_avg(present_csats), 2),
            "absent_avg_csat": round(safe_avg(absent_csats), 2),
            "present_resolve_rate": round(safe_rate(present_resolved), 2),
            "absent_resolve_rate": round(safe_rate(absent_resolved), 2),
            "present_repeat_rate": round(safe_rate(present_repeat), 2),
            "absent_repeat_rate": round(safe_rate(absent_repeat), 2),
        }

    return table


# ═══════════════════════════════════════════════════════════════════════
# PART 3 — impact ranking + behavior combos
# ═══════════════════════════════════════════════════════════════════════

def compute_behavior_impact(calls, detections):
    table = behavior_outcome_table(calls, detections)
    results = []

    for behavior, t in table.items():
        results.append({
            "behavior": behavior,
            "csat_uplift": round(t["present_avg_csat"] - t["absent_avg_csat"], 2),
            "resolve_uplift": round(t["present_resolve_rate"] - t["absent_resolve_rate"], 2),
            "repeat_reduction": round(t["absent_repeat_rate"] - t["present_repeat_rate"], 2),
            "present_count": t["present_count"],
            "absent_count": t["absent_count"],
            "statistically_meaningful": t["present_count"] >= 3 and t["absent_count"] >= 3,
        })

    results.sort(key=lambda x: -x["csat_uplift"])
    return results


def find_behavior_combinations(calls, detections, min_combo_size=2, max_combo_size=3):
    behavior_names = list(BEHAVIORS.keys())
    results = []

    for size in range(min_combo_size, max_combo_size + 1):
        for combo in combinations(behavior_names, size):
            # Find calls where ALL behaviors in the combo are present
            matching_calls = []
            for call in calls:
                det = detections[call.call_id]
                if all(det[b] for b in combo):
                    matching_calls.append(call)

            if len(matching_calls) >= 2:  # need at least 2 calls
                avg_csat = sum(c.outcome["csat"] for c in matching_calls) / len(matching_calls)
                results.append({
                    "behaviors": list(combo),
                    "call_count": len(matching_calls),
                    "avg_csat": round(avg_csat, 1),
                })

    results.sort(key=lambda x: (-x["avg_csat"], -x["call_count"]))
    return results[:5]


# ═══════════════════════════════════════════════════════════════════════
# PART 4 — agent profiles + coaching
# ═══════════════════════════════════════════════════════════════════════

def agent_behavior_profile(calls, detections):
    agent_calls = defaultdict(list)
    for call in calls:
        agent_calls[call.agent_id].append(call)

    profiles = {}
    for agent_id, acalls in agent_calls.items():
        csats = [c.outcome["csat"] for c in acalls]
        resolved = [c.outcome["resolved"] for c in acalls]

        # Behavior rates: what % of this agent's calls include each behavior
        behavior_rates = {}
        for behavior in BEHAVIORS:
            hits = sum(1 for c in acalls if detections[c.call_id][behavior])
            behavior_rates[behavior] = round(hits / len(acalls), 2)

        profiles[agent_id] = {
            "total_calls": len(acalls),
            "avg_csat": round(sum(csats) / len(csats), 1),
            "behavior_rates": behavior_rates,
            "resolve_rate": round(sum(resolved) / len(resolved), 2),
        }

    return profiles


def generate_coaching(profiles, impact_ranking, team_avg_rates=None):
    # Compute team averages if not provided
    if team_avg_rates is None:
        team_avg_rates = {}
        for behavior in BEHAVIORS:
            rates = [p["behavior_rates"][behavior] for p in profiles.values()]
            team_avg_rates[behavior] = sum(rates) / len(rates)

    # Get top impactful behaviors (only statistically meaningful ones)
    top_behaviors = [r for r in impact_ranking if r["statistically_meaningful"]]

    coaching = {}
    for agent_id, profile in profiles.items():
        recs = []

        for impact in top_behaviors:
            beh = impact["behavior"]
            agent_rate = profile["behavior_rates"][beh]
            team_rate = team_avg_rates[beh]

            if agent_rate < team_rate - 0.1:
                # Agent is below average on a high-impact behavior
                recs.append(
                    f"Try using '{beh.replace('_', ' ')}' more often — "
                    f"you use it {agent_rate:.0%} vs team avg {team_rate:.0%}. "
                    f"This behavior is linked to a {impact['csat_uplift']:+.1f} CSAT uplift."
                )
            elif agent_rate > team_rate + 0.1:
                # Agent is above average — reinforce it
                recs.append(
                    f"Great job with '{beh.replace('_', ' ')}' — "
                    f"you use it {agent_rate:.0%} vs team avg {team_rate:.0%}. Keep it up!"
                )

            if len(recs) >= 3:
                break

        # If no specific recs, give general feedback
        if not recs:
            if profile["avg_csat"] >= 4:
                recs.append("Strong overall performance. Maintain your current approach.")
            else:
                recs.append("Focus on professional greetings and empathy statements — these have the highest CSAT impact.")

        coaching[agent_id] = recs

    return coaching


# ═══════════════════════════════════════════════════════════════════════
# RUNNER
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("SIMPLE SOLUTION — Behavior Detection & Outcome Linking")
    print("=" * 60)

    # Part 1
    print(f"\n🔍 Part 1: Detection")
    detections = detect_all(CALLS)
    for call in CALLS[:4]:
        behaviors = detections[call.call_id]
        present = [b for b, v in behaviors.items() if v]
        absent = [b for b, v in behaviors.items() if not v]
        print(f"  {call.call_id} ({call.agent_id}): {len(present)}/{len(BEHAVIORS)}")
        print(f"    ✓ {', '.join(present)}")

    # Part 2
    print(f"\n📊 Part 2: Outcome Table")
    table = behavior_outcome_table(CALLS, detections)
    print(f"  {'Behavior':<25s} {'#Yes':>5s} {'CSAT+':>6s} {'CSAT-':>6s} {'Uplift':>7s}")
    for beh in sorted(table.keys(), key=lambda b: -(table[b]["present_avg_csat"] - table[b]["absent_avg_csat"])):
        t = table[beh]
        delta = t["present_avg_csat"] - t["absent_avg_csat"]
        print(f"  {beh:<25s} {t['present_count']:>4d}  {t['present_avg_csat']:>5.1f}  {t['absent_avg_csat']:>5.1f}  {delta:>+5.1f}")

    # Part 3
    print(f"\n📈 Part 3: Impact Ranking")
    impact = compute_behavior_impact(CALLS, detections)
    for i, row in enumerate(impact[:7]):
        flag = "✓" if row["statistically_meaningful"] else "~"
        print(f"  {i+1}. [{flag}] {row['behavior']:<25s} CSAT:{row['csat_uplift']:+.2f} "
              f"resolve:{row['resolve_uplift']:+.0%} repeat:{row['repeat_reduction']:+.0%}")

    combos = find_behavior_combinations(CALLS, detections)
    print(f"\n  Top combos:")
    for combo in combos[:3]:
        print(f"    {' + '.join(combo['behaviors'][:3])} → CSAT {combo['avg_csat']} ({combo['call_count']} calls)")

    # Part 4
    print(f"\n🎯 Part 4: Coaching")
    profiles = agent_behavior_profile(CALLS, detections)
    coaching = generate_coaching(profiles, impact)
    for agent_id, recs in sorted(coaching.items()):
        csat = profiles[agent_id]["avg_csat"]
        print(f"\n  {agent_id} (CSAT: {csat}):")
        for rec in recs:
            print(f"    → {rec}")

    print("\n" + "=" * 60)